1.2 Enviroment

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

1.3 RAG (Concept of..)

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
llm('hey whats up')

'Hey — not much, just here and ready to help. What’s up with you?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—often you can, but it depends on the course’s enrollment policy and where you are in the schedule.

A few possibilities:
- **Open enrollment:** you can join anytime.
- **Cohort-based course:** you may need to wait for the next start date.
- **In-progress course:** the instructor may allow late entry, but you might have missed earlier material or deadlines.

If you want, I can help you draft a quick message to the instructor or course support asking:
- whether late enrollment is allowed,
- how to catch up,
- and whether you’ll still get full access/certification.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


def rag(question):

    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

1.4 Dataset (forming the knowledge base)

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [ ]:
courses_raw

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [6]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

1.5 Search (+Indexing)

CONCEPTS:

-Searching narrows down the context you are feeding to the llm - less token more accuraracy.

-To search better, you need to index your knowledge base ("documents" variable) first

-Indexing: "organize" your knowledge base into keyword fields and text fields.
 Keyword fields for absolute filtering. 
 Text fields for text search.

-Search: 2 kinds, 
 text/lexical: # of words that matches
 vector/semantic: "meaning", done by llm.



STEPS:

-Create "index" variable

-Index our knowledgebase

-Create "search" function with "boosting field" and "filter"

In [11]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [12]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [13]:
search_results = search(question)

1.6: Building Prompt

CONCEPT:

-Prompt for llms consists of [Instruction][User Prompt]

-[User Prompt]=The bit that changes everytime(query). 
In our example, this would be the "question" that users ask, 

and "context"-which is the narrowed-down information we retrieve from our knowledge base, to reduce the workload of the llm.


STEPS:

-Define datatype USER_PRMPT_TEMPLATE.

-Build function "build_context" to call "search" built in module 1.5, then convert the retrieved data into a FORM EASY FOR LLM o read.

-Build Function "build_prompt", calls "built_context", put this and the "question" input inside the USER_PROMPT_TEMPLATE datatype.


In [14]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [15]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [16]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [17]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [18]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

1.7 LLM

CONCEPT:

-different LLM/API produces different outpts.
-we use OPENAI's "response" API. The response (output) is Pydantic - matches a preset RULE / DATA TYPE. 

-It can contain interesting/useful info like how much token you used!!

-A PROPER PROMPT is consisted of INSTRUCTIONS and PROMPT. 
 We need to include the message history in the prompt - otherwise LLM wont work.

 e.g: if we frist say "we live in Auckland", then ask "what's the weather like?" - if we dont include the history, llm wont know we are asking about weather in Auckland.

 -Seperating INSTRUCTIONS from PROMPTS also prevent PROMPT INJECTION ATTACKS, where hackers literally give instruction in the prompt to ignore security rules.

STEPS:

-Create datatype "message_history"

-Create the function "llm", which creats the data package in the format OPEN AI likes, 
 INSTRUCTION, and PROMPT based on "message_history"

-Finally, we create "rag", which is our entire RAG flow:
 -SEARCH: narrowed down context from indexed knowledge base
 -PROMPT: user question + SEARCH formated into LLM-prefered style + message_history
 -LLM: wraps INSTRUCTION + PROMPT into OPEN-AI data capsule, makes query to OPEN-AI and tell it which LLM model we want to use. 


In [19]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [20]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [21]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [22]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join and start learning now. If you want a certificate, though, you need to submit your project while submissions are still open.


In [23]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.\n\nSelf-paced mode does not offer certificates, because you need to peer-review 3 capstones during the running course. Also, homework is not required for the certificate.'

1.8: RAGE-helper

CONCEPT:

STEPS: